# Lilly Read — clean shipped-model evaluation

Eval only: shipped PaddleOCR PP-OCRv6 medium at confidence floor 0.9 through
`app.ocr.scan`, on 40 hash-pinned original-resolution Commons photographs.
Cohorts were frozen by human visual inspection before inference. No training,
install, product-default change, or publication.


In [ ]:
JOB = "read-clean-eval"
import hashlib, json, os, shutil, subprocess, sys, urllib.error, urllib.request, zipfile
from pathlib import Path
import torch
assert torch.cuda.is_available(), "Kaggle GPU is required for this heavy evaluation"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
GPU = torch.cuda.get_device_name(0)
print(torch.cuda.device_count(), "GPU(s), using", GPU)

def reachable(url):
    try:
        urllib.request.urlopen(url, timeout=20).close()
    except urllib.error.HTTPError:
        pass
    except Exception as exc:
        raise SystemExit(f"network unavailable for {url}: {exc}")
for host in ("https://github.com", "https://pypi.org", "https://huggingface.co"):
    reachable(host)

TEE = Path("/kaggle/working/stdout.txt")
# Offload writes /kaggle/working/experiment_log.json beside the evidence zip.
TEE.parent.mkdir(parents=True, exist_ok=True)
def run(*cmd, quiet=False, env=None):
    line = "$ " + " ".join(str(x) for x in cmd)
    print(line, flush=True)
    with TEE.open("a", encoding="utf-8") as sink:
        sink.write(line + "\n")
        child = subprocess.Popen([str(x) for x in cmd], stdout=subprocess.PIPE,
                                 stderr=subprocess.STDOUT, text=True, bufsize=1,
                                 env={**os.environ, **(env or {})})
        for output in child.stdout:
            if not quiet:
                print(output, end="", flush=True)
            sink.write(output)
        code = child.wait()
    if code:
        raise subprocess.CalledProcessError(code, cmd)


In [ ]:
EXPECTED_GIT_COMMIT = "__LAUNCHER_GIT_COMMIT__"
SCRATCH = Path("/kaggle/temp") if Path("/kaggle/temp").is_dir() else Path("/tmp")
CLONE = SCRATCH / "Lilly"
if CLONE.exists():
    shutil.rmtree(CLONE)
run("git", "clone", "-q", "https://github.com/ssaaffaakk/Lilly.git", str(CLONE))
os.chdir(CLONE)
run("git", "checkout", "-q", EXPECTED_GIT_COMMIT)
got_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
if got_commit != EXPECTED_GIT_COMMIT:
    raise SystemExit(f"clone is {got_commit}, expected {EXPECTED_GIT_COMMIT}")
sys.path.insert(0, str(CLONE))
from training.kaggle_offload import Offload
OFF = Offload(JOB, os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "manual"))
OFF.hardware(GPU)
print("exact code commit", got_commit)


In [ ]:
run(sys.executable, "-m", "pip", "install", "-q", "paddlepaddle-gpu==3.3.1",
    "-i", "https://www.paddlepaddle.org.cn/packages/stable/cu126/")
run(sys.executable, "-m", "pip", "install", "-q", "paddleocr==3.7.0",
    "easyocr==1.7.2", "opencv-contrib-python==4.10.0.84",
    "opencv-python-headless==4.10.0.84", "pillow==12.3.0")
import cv2, paddle, paddleocr
if not paddle.device.is_compiled_with_cuda() or paddle.device.cuda.device_count() < 1:
    raise SystemExit("paddlepaddle-gpu has no CUDA")
if paddle.__version__ != "3.3.1" or paddleocr.__version__ != "3.7.0" or cv2.__version__ != "4.10.0":
    raise SystemExit(f"OCR runtime drift: paddle={paddle.__version__}, "
                     f"paddleocr={paddleocr.__version__}, cv2={cv2.__version__}")
print("OCR runtime", paddle.__version__, paddleocr.__version__, cv2.__version__)


In [ ]:
MANIFEST = CLONE / "training/clean-eval/ocr-commons-40.tsv"
SOURCE = CLONE / "training/clean-eval/ocr-source.json"
source = json.loads(SOURCE.read_text(encoding="utf-8"))
if hashlib.sha256(MANIFEST.read_bytes()).hexdigest() != source["manifest_sha256"]:
    raise SystemExit("committed OCR manifest does not match source record")
PHOTOS = SCRATCH / "ocr-commons-originals"
if PHOTOS.exists():
    shutil.rmtree(PHOTOS)
run(sys.executable, "training/fetch_pinned_ocr_photos.py", "--manifest", str(MANIFEST),
    "--out", str(PHOTOS))
if len(list(PHOTOS.iterdir())) != 40:
    raise SystemExit("OCR source fetch is not exactly 40 originals")
# Exact shipped Paddle detector/recogniser bytes, attached as a private Kaggle
# dataset. The notebook does not trust the dataset's own manifest: the six
# expected hashes are fixed in committed code.
OCR_WEIGHT_HASHES = {
    "PP-OCRv6_medium_det/inference.pdiparams": "85218d2e3d98f5a21c58b4220627be923a97aee5db3cc71f39536ab31ac53960",
    "PP-OCRv6_medium_det/inference.yml": "7298d5ead546584af2504d03355f881ac7a7bc0eb1e282d3e159277c1d0af871",
    "PP-OCRv6_medium_det/inference.json": "0f1a7ec35da36173529c7a60238b7f7919e3831929c3f700ad90ad4896adecd5",
    "PP-OCRv6_medium_rec/inference.pdiparams": "1b01c79a914587933f615569e75de54f2e638ebb5d3f3b3c1b38c24ede8c7319",
    "PP-OCRv6_medium_rec/inference.yml": "991b700facf5b50a7de193468207d5f4255b538dde0d312ae3b7c7a9b6873129",
    "PP-OCRv6_medium_rec/inference.json": "0b2e25e990bd072f1bf77d59d67d508bce6c4bd44af6624e0fb27d6da2cd00e8",
}
weight_manifests = list(Path("/kaggle/input").rglob("weights-sha256.json"))
if len(weight_manifests) != 1:
    raise SystemExit(f"need one attached lilly-ocr-ppocrv6-shipped dataset, found {weight_manifests}")
attached = weight_manifests[0].parent
PADDLE_CACHE = SCRATCH / "paddlex-cache" / "official_models"
for relative, expected in OCR_WEIGHT_HASHES.items():
    source_weight = attached / relative
    if not source_weight.is_file() or hashlib.sha256(source_weight.read_bytes()).hexdigest() != expected:
        raise SystemExit(f"attached shipped OCR weight mismatch: {relative}")
    target_weight = PADDLE_CACHE / relative
    target_weight.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source_weight, target_weight)
print("six shipped Paddle files verified and staged", PADDLE_CACHE)
OFF.metric("ocr_photos", 40, stage="data")
OFF.metric("manifest_sha256", source["manifest_sha256"], stage="data")


In [ ]:
os.environ.update({"PADDLE_PDX_MODEL_SOURCE": "huggingface", "LILLY_READER": "paddle",
                   "LILLY_PADDLE_VERSION": "PP-OCRv6", "LILLY_PADDLE_REC_THRESH": "0.9",
                   "PADDLE_PDX_CACHE_HOME": str(PADDLE_CACHE.parent)})
for forbidden in ("LILLY_PADDLE_REC_DIR", "LILLY_PADDLE_CYRILLIC_RESCUE",
                  "LILLY_PADDLE_DET_SIDE_LEN"):
    os.environ.pop(forbidden, None)
from app import ocr
identity = ocr.reader_identity()
expected = "paddle:PP-OCRv6_medium_det+PP-OCRv6_medium_rec:3.7.0:rec>=0.9"
if identity != expected:
    raise SystemExit(f"reader is {identity}, expected {expected}")
print("shipped reader", identity)
CACHE = Path("/kaggle/working/ocr-reader-output.json")
RAW_JSON = Path("/kaggle/working/ocr-overall.json")
run(sys.executable, "training/evaluate_ocr.py", "--truth", "data/ocr/real-photos/truth.json",
    "--photos", str(PHOTOS), "--cache", str(CACHE),
    "--out", "/kaggle/working/ocr-overall.md", "--json", str(RAW_JSON))
raw = json.loads(RAW_JSON.read_text())
if raw["photographs"] != 28 or raw["reader_context"]["treatment"] != "shipped-scan-2mp-cap":
    raise SystemExit(f"OCR did not score the exact shipped path: {raw}")
COHORT_JSON = Path("/kaggle/working/ocr-legibility.json")
run(sys.executable, "training/ocr_legibility_report.py",
    "--truth", "data/ocr/real-photos/truth.json", "--manifest", str(MANIFEST),
    "--cache", str(CACHE), "--json", str(COHORT_JSON),
    "--markdown", "/kaggle/working/ocr-legibility.md")
cohorts = json.loads(COHORT_JSON.read_text())
if cohorts["reader"] != raw["reader"] or cohorts["all_40"]["photographs"] != 40:
    raise SystemExit("OCR cohort report does not cover the exact 40-photo reading cache")
OFF.metric("ocr_reader_fingerprint", raw["reader"], stage="eval")
OFF.metric("ocr_legible_recall", cohorts["legible_overall"]["recall"], stage="eval")


In [ ]:
OFF.check_trainproof(TEE)
files = [CACHE, RAW_JSON, COHORT_JSON, Path("/kaggle/working/ocr-overall.md"),
         Path("/kaggle/working/ocr-legibility.md"), MANIFEST, SOURCE]
for path in files:
    if not path.is_file() or path.stat().st_size == 0:
        raise SystemExit(f"missing result {path}")
ZIP = Path("/kaggle/working/lilly-read-clean-eval.zip")
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in files:
        archive.write(path, path.name)
if ZIP.stat().st_size < 10_000:
    raise SystemExit(f"result zip too small: {ZIP.stat().st_size}")
OFF.finish("complete", [ZIP.name])
print("wrote", ZIP, ZIP.stat().st_size, "bytes; eval only, no training or default change")
